## How to run in Colab
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
2. Download the datasets first and save them in `C:\Users\sivan\Datasets`:
   - Exercise 1: click [New York City Airbnb Open Data.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/New%20York%20City%20Airbnb%20Open%20Data.zip). It downloads `New York City Airbnb Open Data.zip`.
   - Exercise 2: click [World Happiness Report Dataset.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/World%20Happiness%20Report%20Dataset.zip) and [Global Health and Population Stats.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/Global%20Health%20and%20Population%20Stats.zip). They download `World Happiness Report Dataset.zip` and `Global Health and Population Stats.zip`.
   - Exercise 3: this link opens a GitHub **page**, not the file: [Shop Customer Dataset.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/blob/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20%26%20T/Shop%20Customer%20Dataset.zip). On that page click the **Download** button (the arrow ⬇ at the top right of the file box, "Download raw file"). It downloads `Shop Customer Dataset.zip`.
3. In each exercise, the upload cell asks for its file(s): choose them from `C:\Users\sivan\Datasets`:
   - Exercise 1: `New York City Airbnb Open Data.zip`
   - Exercise 2: `World Happiness Report Dataset.zip` and `Global Health and Population Stats.zip` (you can select both at once with Ctrl+click)
   - Exercise 3: `Shop Customer Dataset.zip`

**Run the cells from top to bottom** (Shift+Enter runs one cell). After a restart, run them again from the top.

# What you will learn

- Perform complex data cleaning and feature engineering on a real-world dataset.
- Integrate and transform multiple datasets for comprehensive analysis.
- Perform Exploring Dimensionality Reduction Techniques.

# What you will create

- A cleaned and enhanced version of the New York City Airbnb dataset with missing values handled using advanced imputation techniques.
- An exploratory data analysis report showcasing the correlations between the newly created features and the target variable.
- Application of PCA for dimensionality reduction, preserving critical information.
- Application of an additional dimensionality reduction technique, such as t-SNE or LDA, with a comparison to PCA results.

## Setup
Run this cell first. It imports the libraries used in all the exercises.

In [ ]:
import pandas as pd
# pandas: for working with tables (DataFrames)
import numpy as np
# numpy: for math on whole columns
import matplotlib.pyplot as plt
# matplotlib: for drawing charts
import seaborn as sns
# seaborn: for nicer statistical charts (built on matplotlib)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# the two scalers we use in Exercises 2 and 3
from sklearn.decomposition import PCA
# PCA: reduces many columns to a few new ones

---
# Exercise 1: Advanced Data Cleaning and Feature Engineering

**Instructions**

Dataset: Use the [New York City Airbnb Open Data](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/New%20York%20City%20Airbnb%20Open%20Data.zip).

- Load the NYC Airbnb dataset.
- Identify and handle missing values in multiple columns using advanced imputation techniques.
- Detect and treat outliers in key columns like ‘price’ and ‘number_of_reviews’.
- Create new features based on existing data, such as ‘booking_rate’ (number of reviews divided by availability) and ‘price_per_person’ (price divided by the number of accommodated guests).
- Perform exploratory data analysis to understand correlations between newly created features and the target variable.

*Hint: For advanced data cleaning techniques, refer to this article on [Data Cleaning](https://towardsdatascience.com/how-to-clean-your-data-in-python-8f178638b98d).*

### Step 1: Load the data

In [ ]:
from google.colab import files
# the Colab tool for uploading files from your computer
uploaded = files.upload()
# choose "New York City Airbnb Open Data.zip" (saved in a variable so its contents are not printed)
!unzip "New York City Airbnb Open Data.zip"
# unzip it: you get the folder "New York City Airbnb Open Data" with "AB_NYC_2019.csv"

In [ ]:
airbnb = pd.read_csv('New York City Airbnb Open Data/AB_NYC_2019.csv')
# load the listings into a DataFrame called airbnb
print(airbnb.shape)
# (number of rows, number of columns)
print(airbnb.columns.tolist())
# all the column names

**Column check:** the print shows **48895 listings and 16 columns**. The columns `price` and `number_of_reviews` exist with these names. There is **no `accommodates` column** (no column with the number of guests), which matters for `price_per_person` in Step 6.

### Step 2: Find the missing values

In [ ]:
print(airbnb.isnull().sum())
# count empty cells in each column

In [ ]:
no_reviews = airbnb['number_of_reviews'] == 0
# True for listings that never got a review
print('Listings with 0 reviews:', no_reviews.sum())
# how many listings have no review
print('Missing reviews_per_month:', airbnb['reviews_per_month'].isnull().sum())
# how many reviews_per_month are empty
print('Missing reviews_per_month among listings with 0 reviews:', airbnb.loc[no_reviews, 'reviews_per_month'].isnull().sum())
# how many of the empty ones belong to listings with 0 reviews

In [ ]:
print('Listings with price 0:', (airbnb['price'] == 0).sum())
# a price of 0 dollars is not a real price, so it is a hidden missing value

**What is missing:** the print shows `name` **16**, `host_name` **21**, `last_review` **10052** and `reviews_per_month` **10052** missing. The second print proves that all **10052** missing `reviews_per_month` belong to the **10052** listings with 0 reviews: they are missing because there was nothing to count, so the right value is **0**. `last_review` is a date that does not exist for these listings, so we leave it empty (we do not use it). We also found **11** listings with a price of **0**, which is not a real price, so we treat them as missing too.

### Step 3: Handle the missing values

In [ ]:
airbnb['reviews_per_month'] = airbnb['reviews_per_month'].fillna(0)
# no review -> 0 reviews per month
airbnb['name'] = airbnb['name'].fillna('Unknown')
# a listing with no name gets the text 'Unknown'
airbnb['host_name'] = airbnb['host_name'].fillna('Unknown')
# a host with no name gets the text 'Unknown'

**Advanced imputation for price (KNNImputer, from the lesson):** a price of 0 is a hidden missing value. Instead of one median for the whole city, we use `KNNImputer(n_neighbors=2)`, as shown in class. For every listing with a missing price, it looks for the **2 most similar listings** (the 2 "nearest neighbours") using the other number columns we give it: `latitude`, `longitude`, `minimum_nights`, `number_of_reviews` and `availability_365`. Then it fills the missing price with the **average price of those 2 listings**. KNNImputer works only with numbers, so we give it these number columns and not the text columns.

In [ ]:
airbnb['price'] = airbnb['price'].replace(0, np.nan)
# turn the price 0 into a real missing value (NaN)
was_zero = airbnb['price'].isnull()
# remember which rows had price 0 (True = now missing), so we can look at them after filling

In [ ]:
from sklearn.impute import KNNImputer
# KNNImputer: fills a missing value with the average of the most similar rows
knn_cols = ['price', 'latitude', 'longitude', 'minimum_nights', 'number_of_reviews', 'availability_365']
# KNNImputer works only with numbers, so we give it price and 5 number columns that describe a listing
imputer = KNNImputer(n_neighbors=2)
# use the 2 most similar listings (the 2 nearest neighbours), as in the lesson
df_imputed = imputer.fit_transform(airbnb[knn_cols])
# fill the missing prices; the result is a table of numbers with the same 6 columns
airbnb['price'] = df_imputed[:, 0]
# column 0 is price (the first in knn_cols), so we write the filled prices back into airbnb
print('Price 0 left:', (airbnb['price'] == 0).sum())
# check: no price of 0 left
print('Missing price left:', airbnb['price'].isnull().sum())
# check: no missing price left
print(airbnb.loc[was_zero, ['neighbourhood_group', 'room_type', 'price']])
# the 11 listings that had price 0, with their new (imputed) price

In [ ]:
print(airbnb.isnull().sum())
# every column except last_review should now show 0

**Explanation:** the last table of the KNNImputer cell shows the **11** listings that had a price of 0 and the new price each one got, for example **47.5** for a private room in Brooklyn and **379.5** for an entire home in Manhattan. Each new price is the **average price of the 2 most similar listings**. KNNImputer only compares the number columns we gave it, so it does not know the `room_type` (text), and some guesses are high for their room type, like **420** for a private room in Brooklyn. Also, the columns are not scaled, so the columns with big numbers (`availability_365`, `minimum_nights`, `number_of_reviews`) count more than `latitude` and `longitude` when it decides which listings are "similar". The checks print **0** price-0 and **0** missing prices left, and the last print shows **0** missing values in every column except `last_review` (which we do not use).

### Step 4: Outliers in price and number_of_reviews

In [ ]:
print(airbnb[['price', 'number_of_reviews']].describe())
# min, max, mean and quartiles of the two columns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
# one row of 2 charts
sns.boxplot(x=airbnb['price'], ax=axes[0])
# boxplot of price
axes[0].set_title('price - before')
# chart title
sns.boxplot(x=airbnb['number_of_reviews'], ax=axes[1])
# boxplot of number_of_reviews
axes[1].set_title('number_of_reviews - before')
# chart title
plt.tight_layout()
# leave space between the charts
plt.show()
# show the charts

In [ ]:
for col in ['price', 'number_of_reviews']:
# repeat for both columns
    Q1 = airbnb[col].quantile(0.25)
    # first quartile
    Q3 = airbnb[col].quantile(0.75)
    # third quartile
    IQR = Q3 - Q1
    # the middle half of the data
    upper = Q3 + 1.5 * IQR
    # values above this are outliers
    print(col, '| IQR upper limit:', upper, '| outliers:', (airbnb[col] > upper).sum())
    # show the limit and how many values are above it
    print(airbnb[col].quantile([0.95, 0.98, 0.99, 1.0]))
    # the high quantiles, to choose the capping value

**Choice:** the `describe()` print shows a median price of **106** but a max of **10000**, and a median of **5** reviews but a max of **629**: both have a long tail of very high values (the boxplots show it too). The IQR limits would mark **2974** prices and **6021** review counts as outliers: too many to remove, and many are real (popular listings and expensive homes). So we **cap** the values instead of removing rows. The quantiles show the jump at the end: price **0.98 = 550, 0.99 = 799, max = 10000**; reviews **0.98 = 172, 0.99 = 214, max = 629**. We cap at **0.99**, because 0.99 already removes the extreme tail (10000 → 799) and changes only 1% of the listings.

In [ ]:
price_cap = airbnb['price'].quantile(0.99)
# the 0.99 quantile of price
reviews_cap = airbnb['number_of_reviews'].quantile(0.99)
# the 0.99 quantile of number_of_reviews
airbnb['price'] = airbnb['price'].clip(upper=price_cap)
# every price above the cap becomes the cap
airbnb['number_of_reviews'] = airbnb['number_of_reviews'].clip(upper=reviews_cap)
# every review count above the cap becomes the cap
print('Price cap:', price_cap, '| Reviews cap:', reviews_cap)
# the values used as caps
print(airbnb[['price', 'number_of_reviews']].describe())
# the two columns after capping
print('Rows:', len(airbnb))
# capping keeps every row

**Explanation:** the caps are **799** for price and **214** for number_of_reviews. After capping, the print shows the price max fell from **10000 to 799** and the std from **240.1 to 121.9**; the reviews max fell from **629 to 214**. The median (106 and 5) did not change, and all **48895** rows are still here.

### Step 5: New feature `booking_rate` (with division by zero handled)

We define **booking_rate = number_of_reviews / availability_365**: the reviews a listing got for each day it is open for booking in the year. A listing with more reviews per open day is booked more often.

`availability_365` can be **0**, and dividing by 0 is not possible, so we first count those rows.

In [ ]:
zero_days = airbnb['availability_365'] == 0
# True for listings that are open 0 days a year
print('Listings with availability_365 = 0:', zero_days.sum())
# how many rows would divide by zero

In [ ]:
airbnb['booking_rate'] = airbnb['number_of_reviews'] / airbnb['availability_365'].replace(0, np.nan)
# replace 0 days with NaN before dividing, so those rows get NaN instead of an error or infinity
airbnb['booking_rate'] = airbnb['booking_rate'].fillna(0)
# a listing that is never open cannot be booked, so its booking rate is 0
print(airbnb['booking_rate'].describe())
# the new feature
print('Infinite values:', np.isinf(airbnb['booking_rate']).sum())
# check: no division by zero left

**Explanation:** the print shows **17533** listings with `availability_365 = 0`. We replaced the 0 days with `NaN` before dividing, so those rows got `NaN` instead of a division by zero, and then we set them to **0** (a listing that is never open is not being booked). The check prints **0** infinite values. The median booking rate is **0.006** reviews per open day; the max is **214** because a listing with 214 reviews and only 1 open day gets 214 / 1.

### Step 6: New feature `price_per_person`

**`price_per_person` needs the number of guests, but this dataset has no `accommodates` column** (the next cell prints `False`). The closest real column is **`room_type`**: it is the only column that tells us how big the place is. We estimate the guests from it: **Entire home/apt = 2 people**, **Private room = 1**, **Shared room = 1** (a bed in a shared room is for one person). This is an assumption, not data from the file, so `price_per_person` here is an **estimate**.

In [ ]:
print('accommodates' in airbnb.columns)
# check if the column accommodates exists (False = it does not)
print(airbnb['room_type'].value_counts())
# the 3 room types and how many listings each has

In [ ]:
people_per_room_type = {'Entire home/apt': 2, 'Private room': 1, 'Shared room': 1}
# our guess of how many people stay, for each room type
airbnb['guests_estimate'] = airbnb['room_type'].map(people_per_room_type)
# give each listing the number of people for its room type
airbnb['price_per_person'] = airbnb['price'] / airbnb['guests_estimate']
# price per night divided by the estimated number of people
print(airbnb.groupby('room_type')[['price', 'price_per_person']].median())
# the median price and price per person for each room type

**Explanation:** the print shows the room types: **25409** entire homes, **22326** private rooms and **1160** shared rooms. With our estimate, an entire home (median **160** a night) costs about **80 per person**, which is close to a private room (**70**). So for 2 people, renting an entire home costs about the same per person as a private room each. Because the guest numbers are estimated from `room_type`, this feature would be more exact with a real `accommodates` column.

### Step 7: Exploratory analysis – correlations of the new features with the target (price)

The exercise does not name the **target variable**. For Airbnb listings, the natural thing to predict is the **price** of a night, so we use `price` (after the 0.99 capping) as the target. We check how strongly the new features `booking_rate`, `price_per_person` and `guests_estimate` are related to it, next to some original columns for comparison.

In [ ]:
eda_cols = ['price', 'booking_rate', 'price_per_person', 'guests_estimate', 'number_of_reviews', 'reviews_per_month', 'availability_365', 'minimum_nights']
# the target (price), the 3 new features and some original columns to compare with
correlations = airbnb[eda_cols].corr()
# the correlation of every pair of columns (-1 to 1)
print(correlations['price'].sort_values(ascending=False).round(2))
# the correlation of each column with price, from the strongest positive to the most negative

In [ ]:
print(airbnb.groupby('room_type')[['price', 'booking_rate']].median().round(3))
# the median price and booking rate for each room type

In [ ]:
plt.figure(figsize=(9, 7))
# a bigger chart
sns.heatmap(correlations, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
# colored table of the correlations: red = positive, blue = negative, the numbers are written in each square
plt.title('Correlations: new features and price')
# chart title
plt.show()
# show the chart

**What the correlations show:**
- **price_per_person: 0.88**, the strongest by far. But this is partly built in: price_per_person **is** the price divided by 1 or 2, so it has to follow the price. It does not add new information about why a listing is expensive.
- **guests_estimate: 0.46**, a real, medium link: listings for more people (entire homes, estimate 2) cost more. The room-type print confirms it: a median of **160** for an entire home vs **70** for a private room and **45** for a shared room.
- **booking_rate: -0.02**, almost **no relationship** with price: cheap and expensive listings are reviewed per open day at about the same rate (median **0.008** vs **0.006**).
- The original columns are weak too: availability_365 **0.12**, minimum_nights **0.03**, number_of_reviews **-0.07**, reviews_per_month **-0.06**.

The heatmap shows the same thing: one dark-red square (price with price_per_person), and in the price row almost white (near 0) squares for the review and availability columns. **Conclusion:** of the new features, only the size of the place (guests_estimate / room type) explains the price. The booking rate tells us about popularity, not about price.

---
# Exercise 2: Complex Data Integration and Transformation

**Instructions**

Datasets: [World Happiness Report](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/World%20Happiness%20Report%20Dataset.zip) and [Global Health and Population Statistics](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/Global%20Health%20and%20Population%20Stats.zip)

- Load both datasets.
- Perform data integration by merging the two datasets on the ‘Country’ column.
- Transform the integrated dataset by normalizing numerical columns like ‘GDP per Capita’ and ‘Life Expectancy’.
- Apply PCA for dimensionality reduction while preserving significant information.
- Conduct a comparative analysis pre- and post-transformation to evaluate the impact of these processes on the data.

*Hint: For guidance on data integration and transformation, check out this article on [Data Transformation](https://medium.com/@consoleflare/methods-of-data-transformation-in-data-mining-462ac9a40e24).*

### Step 1: Load the two datasets

In [ ]:
uploaded = files.upload()
# choose BOTH "World Happiness Report Dataset.zip" and "Global Health and Population Stats.zip" (Ctrl+click)
!unzip "World Happiness Report Dataset.zip"
# unzip it: you get the folder "World Happiness Report Dataset" with 2015.csv to 2019.csv
!unzip "Global Health and Population Stats.zip"
# unzip it: you get the folder "Global Health and Population Stats" with 39 CSV files

In [ ]:
happiness = pd.read_csv('World Happiness Report Dataset/2015.csv')
# the 2015 happiness report: one row per country
print(happiness.shape)
# rows and columns
print(happiness.columns.tolist())
# the column names

In [ ]:
life = pd.read_csv('Global Health and Population Stats/lifeExpectancyAtBirth.csv')
# life expectancy at birth for many countries and years
print(life.shape)
# rows and columns
print(life.columns.tolist())
# the column names
print(life.head())
# the first rows
print(life['Period'].unique())
# the years in the file
print(life['Dim1'].unique())
# the groups in Dim1

**Key columns:** the happiness file has a **`Country`** column (158 countries), but the life expectancy file has no `Country` column: its country column is called **`Location`**. The prints also show that this file has several rows per country: years **2019, 2015, 2010, 2000** (and one strange year, 1920, that we do not use) and **Both sexes / Male / Female**. So before the merge we keep only **2015** (the year of the happiness file) and **Both sexes**, and we rename `Location` to `Country` (and `First Tooltip`, the value, to `Life Expectancy`). The print shows **183** countries, one row each.

In [ ]:
life_2015 = life[(life['Period'] == 2015) & (life['Dim1'] == 'Both sexes')]
# keep only the year 2015 (the same year as the happiness file) and 'Both sexes' (one row per country)
life_2015 = life_2015[['Location', 'First Tooltip']]
# keep the country and the value
life_2015 = life_2015.rename(columns={'Location': 'Country', 'First Tooltip': 'Life Expectancy'})
# rename: Location -> Country (the same key name as in happiness), First Tooltip -> Life Expectancy
print(life_2015.shape)
# one row per country
print(life_2015.head())
# the first rows

### Step 2: Merge on Country and check how many countries match

In [ ]:
matched = happiness['Country'].isin(life_2015['Country'])
# True for each happiness country that also appears in the life expectancy table
print('Happiness countries:', len(happiness))
# countries in the happiness file
print('Matched:', matched.sum())
# countries found in both files
print('Not matched:', happiness.loc[~matched, 'Country'].tolist())
# the happiness countries with no match (~ means NOT)

In [ ]:
print(life_2015[life_2015['Country'].str.contains('United')]['Country'].tolist())
# how the health file writes the countries whose name has 'United' in it

In [ ]:
merged = pd.merge(happiness, life_2015, on='Country', how='inner')
# inner merge: keep only the countries that are in both tables
print(merged.shape)
# rows and columns after the merge
print(merged[['Country', 'Happiness Score', 'Economy (GDP per Capita)', 'Life Expectancy']].head())
# the first rows of the merged table

**Explanation:** **134 of the 158** happiness countries match. The 24 that do not match are written differently in the two files, for example "United States", "United Kingdom", "Russia" and "South Korea". The health file uses the long official names: the print with 'United' shows it writes **"United States of America"** and **"United Kingdom of Great Britain and Northern Ireland"**. We keep the 134 exact matches and do not rename the others by hand. The inner merge keeps only the **134** matched countries: the print shows **134 rows and 13 columns** (the 12 happiness columns + Life Expectancy).

### Step 3: Normalize the number columns

In [ ]:
numeric_cols = ['Happiness Score', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Life Expectancy']
# the number columns we use (Happiness Rank, Standard Error and Dystopia Residual are left out, see below)
print(merged[numeric_cols].describe().round(2))
# the columns before scaling: very different ranges

In [ ]:
scaler = StandardScaler()
# create the standard scaler (mean 0, std 1)
scaled = scaler.fit_transform(merged[numeric_cols])
# scale every column
scaled_df = pd.DataFrame(scaled, columns=numeric_cols)
# put the result back in a table with the same column names
print(scaled_df.describe().round(2))
# after scaling: every column has mean 0 and std 1

**Explanation:** we left out **Happiness Rank** (it is just the Happiness Score in order), **Standard Error** (a measurement error, not a feature of the country) and **Dystopia Residual** (a leftover from the score calculation). The first `describe()` shows very different ranges: **Life Expectancy goes from 47.67 to 83.62**, while the other columns are between **0 and 1.69** (Happiness Score 2.84 to 7.59). After `StandardScaler`, the second `describe()` shows **mean 0.00 and std 1.00** for every column. We use StandardScaler because PCA works with variance, and here every column should count the same. Notes on the names in the instructions: the exercise says **‘GDP per Capita’**, which is called **`Economy (GDP per Capita)`** in this file. It says **normalizing**: we use **Z-score normalization** (StandardScaler) rather than Min-Max, because the next step is PCA, which compares variances, so every column should have the same variance (std 1).

### Step 4: PCA, and compare before and after

In [ ]:
pca_unscaled = PCA(n_components=2)
# PCA on the original (not scaled) numbers
pca_unscaled.fit(merged[numeric_cols])
# fit finds the 2 main directions
print('Before scaling - variance kept by PC1, PC2:', pca_unscaled.explained_variance_ratio_.round(3))
# the share of variation each component keeps
print(pd.DataFrame(pca_unscaled.components_.T, index=numeric_cols, columns=['PC1', 'PC2']).round(2))
# how much each column counts in each component

In [ ]:
pca = PCA(n_components=2)
# PCA on the scaled numbers
components = pca.fit_transform(scaled_df)
# each country gets 2 new values: PC1 and PC2
print('After scaling - variance kept by PC1, PC2:', pca.explained_variance_ratio_.round(3))
# the share of variation each component keeps
print('Total:', round(pca.explained_variance_ratio_.sum(), 3))
# both together
print(pd.DataFrame(pca.components_.T, index=numeric_cols, columns=['PC1', 'PC2']).round(2))
# how much each column counts in each component

In [ ]:
merged['PC1'] = components[:, 0]
# the first component of each country
merged['PC2'] = components[:, 1]
# the second component of each country
plt.figure(figsize=(9, 6))
# a bigger chart
sns.scatterplot(data=merged, x='PC1', y='PC2', hue='Region')
# one point per country, colored by its region
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
# put the legend outside the chart
plt.title('Countries after PCA (scaled data)')
# chart title
plt.show()
# show the chart

In [ ]:
print(merged[['PC1', 'Happiness Score']].corr().round(2))
# correlation between PC1 and the happiness score (close to 1 or -1 = strongly related)
print(merged.sort_values('PC1')[['Country', 'PC1', 'Happiness Score']].head(3))
# the 3 countries with the lowest PC1
print(merged.sort_values('PC1')[['Country', 'PC1', 'Happiness Score']].tail(3))
# the 3 countries with the highest PC1

**Comparison before and after scaling:**
- **Before scaling**, PC1 keeps **98.8%** of the variance, but the table shows it is almost only **Life Expectancy (0.99)**. Life Expectancy has the biggest numbers (std 7.73, the others below 1.2), so PCA only "sees" it. The other columns hardly count.
- **After scaling**, PC1 keeps **56.7%** and PC2 **17.2%** (**73.9%** together, 2 columns instead of 8). Now PC1 is a mix of almost all the columns (Happiness 0.43, Economy 0.42, Life Expectancy 0.42, Health 0.41, Family 0.37), and PC2 is mostly **Generosity (0.64), Freedom (0.48) and Trust (0.42)**.

The correlation print shows **PC1 and the Happiness Score have a correlation of 0.93**: PC1 is like a "well-being" score. The lowest PC1 are the Central African Republic, Burundi and Togo, and the highest are Denmark, Norway and Switzerland. The scatter plot shows the same: Sub-Saharan Africa on the left and Western Europe, North America and Australia/New Zealand on the right.

### Step 5: Compare the data before and after the transformation

In [ ]:
key_cols = ['Economy (GDP per Capita)', 'Life Expectancy', 'Happiness Score']
# the two columns the exercise names, and the happiness score
print('Before scaling:')
# label
print(merged[key_cols].corr().round(3))
# correlations between the original columns
print('After scaling:')
# label
print(scaled_df[key_cols].corr().round(3))
# correlations between the scaled columns

In [ ]:
print('Ranges before:')
# label
print(merged[key_cols].agg(['min', 'max', 'std']).round(2))
# the original ranges
print('Ranges after:')
# label
print(scaled_df[key_cols].agg(['min', 'max', 'std']).round(2))
# the scaled ranges
print('Columns before PCA:', scaled_df.shape[1], '| after PCA:', components.shape[1])
# 8 columns became 2

**Pre- and post-transformation comparison:**
- **Relationships are kept:** the correlation tables before and after scaling are **identical**: GDP vs Life Expectancy **0.829**, GDP vs Happiness **0.799**, Life Expectancy vs Happiness **0.755**. Scaling only changes the units, not how the columns move together.
- **Ranges are now comparable:** before, Life Expectancy went from **47.67 to 83.62** (std **7.73**) and GDP from **0.02 to 1.69** (std **0.41**), so Life Expectancy was about 19 times more spread out. After scaling, both have a **std of 1.00** and similar ranges (GDP **-2.05 to 2.08**, Life Expectancy **-3.20 to 1.46**).
- **Impact on PCA (from Step 4):** without scaling, PC1 was almost only Life Expectancy (**98.8%** of the variance, loading **0.99**). After scaling, PC1 mixes all the columns, and 2 components keep **73.9%** of the information.
- **Data reduction:** **8 columns → 2 columns** for 134 countries, and PC1 still has a **0.93** correlation with the happiness score.

---
# Exercise 3 : Exploring Dimensionality Reduction Techniques

**Instructions**

- Read this article : [A Complete Guide On Dimensionality Reduction](https://medium.com/analytics-vidhya/a-complete-guide-on-dimensionality-reduction-62d9698013d2)
- Import this dataset : [Shop Customer Data](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/blob/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20%26%20T/Shop%20Customer%20Dataset.zip)
- Implement Principal Component Analysis (PCA) and observe how much variance is retained with different numbers of components.
- Apply at least one more dimensionality reduction technique (like t-SNE or LDA) and compare its results with PCA.
- Visualize the results of these techniques using plots.
- Write a brief analysis of how dimensionality reduction impacted the dataset and the insights you gained from the visualizations.

**Note:** the first bullet (read the article) is reading, not code, so it has no cell here. The notebook does the other five bullets.

### Step 1: Load and prepare the data

In [ ]:
uploaded = files.upload()
# choose "Shop Customer Dataset.zip"
!unzip "Shop Customer Dataset.zip"
# unzip it: you get the folder "Shop Customer Dataset" with "Customers.csv"

In [ ]:
customers = pd.read_csv('Shop Customer Dataset/Customers.csv')
# load the customers into a DataFrame
print(customers.shape)
# rows and columns
print(customers.head())
# the first rows
print(customers.isnull().sum())
# empty cells in each column

In [ ]:
customers['Profession'] = customers['Profession'].fillna('Unknown')
# the customers with no profession get 'Unknown'
X = customers.drop(columns=['CustomerID'])
# CustomerID is only a number for each customer, so it is not a feature
X = pd.get_dummies(X, columns=['Gender', 'Profession'], dtype=int)
# one-hot encode the 2 text columns
print(X.shape)
# rows and columns after encoding
print(X.columns.tolist())
# the feature names

In [ ]:
scaler = StandardScaler()
# PCA and t-SNE need the columns on the same scale (income is in the thousands, the 0/1 columns are 0 or 1)
X_scaled = scaler.fit_transform(X)
# scale every column to mean 0, std 1
print(X_scaled.shape)
# the same shape, now all scaled

**Explanation:** the first prints show **2000 customers and 8 columns**, with **35** missing `Profession`; we filled them with `'Unknown'`. We removed `CustomerID` (it is only a row number) and one-hot encoded `Gender` and `Profession`, so the table has **17 feature columns**. `StandardScaler` puts all of them on the same scale, because PCA and t-SNE would otherwise be dominated by `Annual Income ($)`, which is in the thousands.

### Step 2: PCA – how much variance is kept with different numbers of components

In [ ]:
pca_full = PCA()
# PCA with all the components (as many as there are columns)
pca_full.fit(X_scaled)
# fit finds all the components
cumulative = np.cumsum(pca_full.explained_variance_ratio_)
# cumsum adds up the shares: 1 component, 1+2 components, 1+2+3 ...
for i in [1, 2, 3, 5, 8, 10, 12]:
# a few different numbers of components
    print(i, 'components keep', round(cumulative[i - 1] * 100, 1), '% of the variance')
    # the share of variance kept with i components (index i-1 because Python starts at 0)

In [ ]:
needed = np.argmax(cumulative >= 0.90) + 1
# the first number of components that keeps at least 90% of the variance
print('Total number of components:', len(cumulative))
# as many as there are columns
print('Components needed for 90% of the variance:', needed)
# how many we need to keep 90%

In [ ]:
plt.plot(range(1, len(cumulative) + 1), cumulative, marker='o')
# line chart: number of components -> share of variance kept
plt.axhline(0.90, color='red', linestyle='--')
# red line at 90%
plt.xlabel('Number of components')
# x axis label
plt.ylabel('Cumulative explained variance')
# y axis label
plt.title('PCA - variance kept by the number of components')
# chart title
plt.show()
# show the chart

**Explanation:** the variance print shows that each component keeps only a small share: **1 component 11.8%, 2 components 19.8%, 5 components 40.6%, 10 components 72.5%, 12 components 84.2%**. To keep **90%** we need **14 of the 17** components, and the line chart rises almost straight, without a clear "elbow". This means the columns are **not strongly related**: no few directions explain most of the customers, so PCA cannot squeeze this data much.

In [ ]:
pca_2 = PCA(n_components=2)
# PCA with 2 components, for a flat chart
pca_points = pca_2.fit_transform(X_scaled)
# 2 new values for every customer
print('Variance kept by 2 components:', round(pca_2.explained_variance_ratio_.sum() * 100, 1), '%')
# how much the chart can show

### Step 3: t-SNE (a second dimensionality reduction technique)

In [ ]:
from sklearn.manifold import TSNE
# t-SNE: places customers that are similar close together on a 2D chart
tsne = TSNE(n_components=2, random_state=42)
# 2 dimensions; random_state=42 gives the same result every run
tsne_points = tsne.fit_transform(X_scaled)
# 2 new values for every customer (this can take a minute)
print(tsne_points.shape)
# one row per customer, 2 columns

### Step 4: Visualize PCA and t-SNE side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# one row of 2 charts
sns.scatterplot(x=pca_points[:, 0], y=pca_points[:, 1], hue=customers['Gender'], alpha=0.6, ax=axes[0])
# PCA: each customer is a point, colored by gender
axes[0].set_title('PCA (2 components)')
# chart title
sns.scatterplot(x=tsne_points[:, 0], y=tsne_points[:, 1], hue=customers['Gender'], alpha=0.6, ax=axes[1])
# t-SNE: the same customers, colored by gender
axes[1].set_title('t-SNE (2 dimensions)')
# chart title
plt.tight_layout()
# leave space between the charts
plt.show()
# show the charts

In [ ]:
print(pd.DataFrame(pca_2.components_.T, index=X.columns, columns=['PC1', 'PC2']).round(2))
# how much each feature counts in PC1 and PC2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# one row of 2 charts
sns.scatterplot(x=pca_points[:, 0], y=pca_points[:, 1], hue=customers['Spending Score (1-100)'], palette='viridis', ax=axes[0])
# PCA colored by spending score
axes[0].set_title('PCA colored by Spending Score')
# chart title
sns.scatterplot(x=tsne_points[:, 0], y=tsne_points[:, 1], hue=customers['Spending Score (1-100)'], palette='viridis', ax=axes[1])
# t-SNE colored by spending score
axes[1].set_title('t-SNE colored by Spending Score')
# chart title
plt.tight_layout()
# leave space between the charts
plt.show()
# show the charts

In [ ]:
combos = customers.groupby(['Gender', 'Profession']).size()
# the number of customers for every gender + profession pair
print('Gender + profession pairs:', len(combos))
# how many different pairs there are
print('Customers in the smallest pair:', combos.min(), '| in the largest pair:', combos.max())
# the size of the smallest and the largest pair

### Step 5: Analysis

**How dimensionality reduction affected the data:**
- **PCA:** 2 components keep only **19.8%** of the variance (14 components are needed for 90%), so the 2D PCA chart hides most of the information. The loadings print shows that **PC1 is just Gender (Female +0.70, Male -0.70)** and **PC2 is mostly Profession_Artist (+0.82)**. That is why the PCA chart shows 4 groups: male / female on the left and right, and artists / non-artists at the top and bottom.
- **t-SNE** keeps the neighbours close and shows many small separate islands, one color each. The last print shows there are **20** gender + profession pairs (2 genders × 10 professions, including Unknown), and the islands match those pairs. So t-SNE separates the groups much more clearly than PCA, but its axes have no meaning (we cannot read "how much" from them).

**Insights from the charts:**
- Both techniques mostly find groups made by the **0/1 encoded columns** (gender and profession), not by the number columns.
- In the charts colored by **Spending Score**, every group has all colors mixed (low and high spenders together). So gender and profession alone **do not explain** the spending score in this data.
- Because of this, reducing these customers to 2 dimensions is good for drawing pictures of the groups, but it loses too much information (80% with PCA) to replace the original 17 columns in a model.